# Step 3 — Data Processing and Analysis

This notebook explores, cleans, and converts the generated datasets into standard training formats. It serves as the bridge between raw LLM outputs and the structured fine-tuning dataset.

## Objectives:
1. **Examine the data shape** of the master clinical annotations.
2. **Normalize and clean** meal/exercise tags in the daily patient logs.
3. **Compute dataset statistics** (unique meals, top meals, exercise events).
4. **Convert the datasets** to ChatML format for fine-tuning.

## 1. Explore Dataset Structure

Let's load the master recommendations dataset and inspect its structure.

In [ ]:
import json

with open('MASTER_DATASET_RECOMMENDATIONS.json', 'r', encoding='utf-8') as f:
    recommendations = json.load(f)

print(f"Total patient records: {len(recommendations)}")

# Inspect the shape of a single patient record
def get_shape(obj):
    if isinstance(obj, dict):
        return {k: get_shape(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [get_shape(obj[0])] if obj else []
    else:
        return type(obj).__name__

print("\nRecord schema:")
print(json.dumps(get_shape(recommendations[0]), indent=2))

## 2. Meal and Activity Normalization

The patient logs sometimes contain slightly corrupted or inconsistently spelled values (e.g., `bagel_creamcheese` vs `bagel_cream_cheese`). Additionally, exercise events are sometimes labeled as meals (`M=jog` instead of `EX=jog`).

Below, we run a normalization script that:
- Maps synonyms to canonical names.
- Moves exercise events out of the meal tag (`M=`) and into an exercise tag (`EX=`).
- Removes numerical or empty corrupted entries.

In [ ]:
import re

synonyms = {
    "bagel_creamcheese": "bagel_cream_cheese",
    "bagel_with_cream_cheese": "bagel_cream_cheese",
    "scrambled_eggs_with_cheese_and_bacon": "eggs_bacon",
    "eggs_and_bacon": "eggs_bacon",
    "spaghetti_carbonara": "pasta_carbonara",
    "chicken_and_rice": "chicken_rice",
    "caesar_salad_with_chicken": "chicken_caesar_salad",
}

ex_set = {
    "basketball", "bike", "cycling", "dance", "jog", "run", "run_5k",
    "spin", "spin_class", "swim", "tennis", "walk", "walk_45min",
    "weights", "yoga"
}

def clean_value(val):
    val = val.strip("_").lower()
    if val.isdigit():
        return None
    if val in ex_set:
        return ("EX", val)
    if val in synonyms:
        val = synonyms[val]
    return ("M", val)

# Clean all records
for item in recommendations:
    new_logs = []
    for day in item["daily_logs"]:
        matches = re.findall(r'M=([a-zA-Z0-9_]+)', day)
        new_day = day
        for m in matches:
            res = clean_value(m)
            if res is None:
                new_day = re.sub(rf'M={m}', '', new_day)
            else:
                typ, val = res
                if typ == "EX":
                    new_day = re.sub(rf'M={m}', f'EX={val}', new_day)
                elif val != m:
                    new_day = re.sub(rf'M={m}', f'M={val}', new_day)
        new_logs.append(new_day)
    item["daily_logs"] = new_logs

print("Logs cleaned successfully.")

## 3. Dataset Statistics

Let's analyze the cleaned dataset to find unique meals, the top 20 most frequent meals, and the frequency of exercise events.

In [ ]:
from collections import Counter

meals = []
total_ex_events = 0

for item in recommendations:
    for day in item["daily_logs"]:
        # Extract meals
        meal_matches = re.findall(r'M=([a-zA-Z0-9_]+)', day)
        meals.extend(meal_matches)
        # Count exercise events
        total_ex_events += len(re.findall(r'EX=[a-zA-Z0-9_]+', day))

meal_counts = Counter(meals)
print(f"Total unique meals: {len(meal_counts)}")
print(f"Total exercise events: {total_ex_events}")

print("\nTop 20 most common meals:")
for meal, count in meal_counts.most_common(20):
    print(f"  {meal} : {count}")

## 4. Convert to ChatML JSONL Format

We convert the master files (`MASTER_DATASET_PATTERNS.json` and `MASTER_DATASET_RECOMMENDATIONS.json`) into ChatML format, which is the standard structure for fine-tuning our Qwen student model using libraries like Unsloth and TRL SFTTrainer.

In [ ]:
import os

def create_user_prompt(patient):
    info = patient.get("basic_info", {})
    logs = patient.get("daily_logs", [])
    
    prompt = (
        f"Patient Profile:\n"
        f"- Age: {info.get('age', 'Unknown')}\n"
        f"- Sex: {info.get('sex', 'Unknown')}\n"
        f"- Diabetes Type: {info.get('diabetes_type', 'Unknown')}\n"
        f"- Weight: {info.get('weight_kg', 'Unknown')} kg\n\n"
        f"Daily Logs:\n"
    )
    for log in logs:
        prompt += f"{log}\n"
    return prompt.strip()

def convert_to_chatml(input_file, output_file, task_type):
    if not os.path.exists(input_file):
        print(f"❌ {input_file} not found. Skipping.")
        return

    print(f"🔄 Converting {input_file} to ChatML...")
    
    if task_type == "PATTERNS":
        system_prompt = (
            "You are an expert AI endocrinology assistant. Analyze the patient's demographics "
            "and timeline logs to identify physiological glycemic patterns. You must respond STRICTLY "
            "with a valid JSON object containing 'feature', 'patterns', and 'message' keys."
        )
    else:
        system_prompt = (
            "You are an expert AI endocrinology assistant. Analyze the patient's demographics "
            "and timeline logs to provide actionable clinical recommendations. You must respond STRICTLY "
            "with a valid JSON object containing 'feature', 'recommendations', and 'message' keys."
        )

    count = 0
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        data = json.load(f_in)
        for patient in data:
            user_content = create_user_prompt(patient)
            assistant_content = json.dumps(patient.get("analysis", {}), ensure_ascii=False)
            
            chatml_record = {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": assistant_content}
                ]
            }
            f_out.write(json.dumps(chatml_record, ensure_ascii=False) + '\n')
            count += 1

    print(f"✅ Saved {count} ChatML records to {output_file}")

# Run conversion
convert_to_chatml("MASTER_DATASET_PATTERNS.json", "train_patterns.jsonl", "PATTERNS")
convert_to_chatml("MASTER_DATASET_RECOMMENDATIONS.json", "train_recommendations.jsonl", "RECOMMENDATIONS")

## 5. Audit ChatML Output

Let's review one resulting ChatML conversation format.

In [ ]:
with open("train_patterns.jsonl", "r", encoding="utf-8") as f:
    first_line = f.readline()
    
print("Sample ChatML record:")
print(json.dumps(json.loads(first_line), indent=2, ensure_ascii=False))